In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

In [1]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy(operational=True, min_num_qubits=16).name

> **Note:** Le Qiskit Functions sono una funzionalità sperimentale disponibile esclusivamente per gli utenti dei piani IBM Quantum&reg; Premium, Flex e On-Prem (tramite IBM Quantum Platform API). Sono in stato di anteprima e soggette a modifiche.

## Panoramica
Nella chimica quantistica, il problema della struttura elettronica consiste nel trovare le soluzioni all'equazione di Schrödinger elettronica — le funzioni d'onda quantistiche che descrivono il comportamento degli elettroni del sistema. Queste funzioni d'onda sono vettori di ampiezze complesse, ciascuna corrispondente al contributo di una possibile configurazione elettronica.

Lo stato fondamentale è la funzione d'onda del sistema a energia minima e riveste un'importanza particolare nello studio dei sistemi molecolari. L'approccio più accurato per calcolare lo stato fondamentale considera tutte le possibili configurazioni elettroniche, ma questo diventa intrattabile per sistemi più grandi, poiché il numero di configurazioni cresce esponenzialmente con la dimensione del sistema.

L'Handover Iterative Variational Quantum Eigensolver (HI-VQE) è un innovativo metodo ibrido quantistico-classico per stimare con precisione lo stato fondamentale di sistemi molecolari. Integra hardware quantistico con elaborazione classica, usando processori quantistici per esplorare efficientemente le configurazioni elettroniche candidate e calcolando la funzione d'onda risultante su computer classici. Generando funzioni d'onda compatte ma chimicamente accurate, HI-VQE potenzia la ricerca e la scoperta nella chimica quantistica e nella scienza dei materiali.

![Immagine che mostra una panoramica dell'algoritmo HI-VQE di Qunova](../docs/images/guides/qunova-chemistry/overview.svg)

HI-VQE riduce la complessità computazionale del problema della struttura elettronica stimando efficientemente lo stato fondamentale con elevata precisione. Si concentra su un sottoinsieme accuratamente selezionato delle configurazioni elettroniche più rilevanti, ottimizzando sia l'accuratezza che l'efficienza.

Combinando i punti di forza dei computer classici e quantistici, HI-VQE affina e migliora iterativamente la stima attuale della funzione d'onda. Le sue tecniche uniche di costruzione del sottospazio contribuiscono a rendere la selezione delle configurazioni più efficiente, offrendo agli utenti un maggiore controllo computazionale e una precisione migliorata nelle simulazioni di chimica quantistica.

Se desideri approfondire l'algoritmo, puoi [leggere il relativo articolo di ricerca](https://arxiv.org/abs/2503.06292).

## Descrizione
Il numero di configurazioni elettroniche per un sistema molecolare cresce esponenzialmente con la dimensione del sistema. Tuttavia, per certi stati elettronici, come lo stato fondamentale, è comune che solo una piccola frazione delle configurazioni contribuisca significativamente all'energia dello stato. I metodi Selected Configuration Interaction (SCI) sfruttano questa sparsità per ridurre i costi computazionali, identificando e concentrandosi sulle configurazioni più rilevanti. Questo sottoinsieme di configurazioni è detto sottospazio.

HI-VQE sfrutta l'efficienza intrinseca dei computer quantistici nella rappresentazione dei sistemi molecolari per supportare la ricerca del sottospazio. Integra subroutine classiche e quantistiche per risolvere il problema della struttura elettronica con elevata precisione. A differenza dei metodi SCI quantistici esistenti, HI-VQE combina addestramento variazionale, costruzione iterativa del sottospazio e screening delle configurazioni tramite pre-diagonalizzazione per migliorare l'efficienza riducendo le misurazioni quantistiche, le iterazioni e i costi di diagonalizzazione classica. HI-VQE può quindi essere applicato a sistemi molecolari più grandi che richiedono più qubit, e riduce il costo per risolvere un problema di una data dimensione allo stesso grado di accuratezza.

![Immagine che mostra una descrizione dettagliata di ogni fase dell'algoritmo HI-VQE di Qunova.](../docs/images/guides/qunova-chemistry/description.avif)

Per calcolare lo stato fondamentale di un sistema, HI-VQE utilizza dapprima il pacchetto di chimica classica PySCF per generare una rappresentazione molecolare a partire dagli input forniti dall'utente, come la geometria molecolare e altre informazioni sul sistema. Entra quindi in un ciclo di ottimizzazione ibrido quantistico-classico, raffinando iterativamente un sottospazio per rappresentare ottimalmente lo stato fondamentale riducendo al minimo il numero di configurazioni incluse. Il ciclo continua fino al soddisfacimento dei criteri di convergenza, come la dimensione del sottospazio o la stabilità dell'energia, dopodiché vengono restituiti la funzione d'onda dello stato fondamentale calcolata e la sua energia. Questi risultati possono essere utilizzati per costruire superfici di energia potenziale accurate ed eseguire ulteriori analisi del sistema.

Il ciclo di ottimizzazione si concentra sull'adeguamento dei parametri di un circuito quantistico per generare un sottospazio di alta qualità. HI-VQE offre tre opzioni di circuito quantistico: [`excitation_preserving`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.excitation_preserving), [efficient_su2](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) e [LUCJ](https://qiskit-community.github.io/ffsim/explanations/lucj.html). L'ottimizzazione viene inizializzata in prossimità dello stato di riferimento Hartree-Fock per la sua idoneità generale. Il circuito viene poi eseguito su un dispositivo quantistico e le configurazioni vengono campionate dallo stato quantistico risultante, prima di essere restituite come stringhe binarie. A causa del rumore del dispositivo quantistico, alcune configurazioni campionate potrebbero essere fisicamente non valide, non conservando il numero di elettroni o lo spin. HI-VQE risolve questo problema usando il processo di configuration recovery del pacchetto [qiskit-addon-sqd](/guides/qiskit-addons-sqd#sample-based-quantum-diagonalization-sqd-overview), in modo che gli utenti possano correggere le configurazioni non valide o scartarle.

Le configurazioni valide subiscono poi un passaggio di screening opzionale per rimuovere quelle che si prevede contribuiscano in misura minima. Questo riduce la dimensione del sottospazio, abbassando così il costo del passaggio di diagonalizzazione. Se lo screening è abilitato, viene costruita un'Hamiltoniana del sottospazio preliminare dalle configurazioni valide e viene eseguita una diagonalizzazione con criteri di terminazione molto approssimati. Sebbene la precisione delle ampiezze risultanti per ciascuna configurazione sia bassa, questo metodo è efficace per prevedere quali configurazioni escludere dal sottospazio in quella iterazione ed è rapido da calcolare.

Le configurazioni selezionate vengono aggiunte al sottospazio e l'Hamiltoniana del sistema viene proiettata in questo sottospazio. Il sottospazio si aggiorna iterativamente, preservando le configurazioni più rilevanti attraverso le iterazioni. Questo approccio si distingue dai metodi alternativi perché il circuito quantistico non deve approssimare lo stato fondamentale completo ad ogni passo.

Successivamente, l'Hamiltoniana del sottospazio viene diagonalizzata classicamente per ottenere il valore proprio minimo e il corrispondente autovettore, che rappresentano un'approssimazione dello stato fondamentale e della sua energia. Con il miglioramento della qualità del sottospazio attraverso le iterazioni, lo stato fondamentale calcolato approssima sempre meglio il vero stato fondamentale. A questo punto può essere eseguito un ulteriore passaggio di screening per rimuovere dal sottospazio le configurazioni che non contribuiscono in modo sostanziale allo stato fondamentale calcolato. Questo passo garantisce che il sottospazio portato nell'iterazione successiva sia il più compatto possibile. La valutazione si basa sulle ampiezze restituite dalla diagonalizzazione, poiché queste rappresentano il contributo di importanza di ciascuna configurazione allo stato fondamentale calcolato.

Un controllo di convergenza determina quindi se ulteriori iterazioni migliorerebbero i risultati. In caso affermativo, viene eseguito un opzionale passo di espansione classica, i parametri del circuito quantistico vengono aggiornati per minimizzare ulteriormente l'energia calcolata e il processo si ripete. Il passo di espansione classica genera configurazioni aggiuntive per il sottospazio, integrando quelle campionate dal dispositivo quantistico. Individua prima la configurazione con l'ampiezza maggiore nei risultati della diagonalizzazione, per poi generare nuove configurazioni con singole e doppie eccitazioni a partire dalla configurazione identificata. Il numero desiderato di queste configurazioni viene poi aggiunto al sottospazio.

Una volta stabilito che le iterazioni sono converge, HI-VQE restituisce lo stato fondamentale calcolato (sotto forma degli stati nel sottospazio e delle loro ampiezze nella funzione d'onda dello stato fondamentale), la sua energia e una misura della varianza dell'energia che indica se lo stato calcolato costituisce un autostato dell'Hamiltoniana del sistema.

Gli utenti possono scegliere il circuito quantistico utilizzato e il numero di shot per ciascun circuito quantistico, nonché controllare la dimensione del sottospazio o abilitare la generazione classica di configurazioni aggiuntive a supporto di quelle generate quantisticamente. In questo modo, gli utenti possono adattare il comportamento di HI-VQE alle loro applicazioni specifiche.

## Per iniziare
Prima di tutto, [richiedi l'accesso alla funzione](https://forms.office.com/r/zN3hvMTqJ1).
Poi, autenticati con la tua [chiave API IBM Quantum&reg;](http://quantum.cloud.ibm.com/) e, avendo già [salvato il tuo account](/guides/functions#install-qiskit-functions-catalog-client) nell'ambiente locale, seleziona la Qiskit Function come segue:

In [ ]:
import reprlib
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("qunova/hivqe-chemistry")

## Input
Consulta la tabella seguente per tutti i parametri di input accettati dalla funzione. Le sezioni successive [Opzioni molecola](#molecule-options) e [Opzioni HI-VQE](#hi-vqe-options) forniscono ulteriori dettagli su quegli argomenti.

| Nome                   | Tipo                                                           | Descrizione                                                                                                                                                                                                                                                                                                 | Obbligatorio | Predefinito                                                                  | Esempio                                                                                   |
|------------------------|----------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|-------------------------------------------------------------------------------------------|
| geometry               | Union[List[List[Union[str, Tuple[float, float, float]]]], str] | Può essere una stringa o liste strutturate contenenti coppie atomo-coordinata. Se fornito come stringa, deve essere una geometria molecolare in formato Cartesian Coordinate. Se fornito come lista, deve essere una lista di liste contenenti ciascuna una stringa atomo e una tupla di coordinate. | Sì      | N/A                                                                      | `[['O', (0, 0, 0)], ['H', (0, 1, 0)], ['H', (0, 0, 1)]]` oppure `"O 0 0 0; H 0 1 0; H 0 0 1"` |
| backend\_name          | str                                                            | Nome del backend su cui eseguire la query.                                                                                                                                                                                                                                                                      | Sì      | N/A                                                                      | `ibm_fez`                                                                                 |
| max\_states            | int                                                            | La dimensione massima del sottospazio per la diagonalizzazione. Verranno usati meno stati se il numero non è un quadrato perfetto.                                                                                                                                                                                                                                                    | Sì      | N/A                                                                      | `100`                                                                                     |
| max\_expansion\_states | int                                                            | Il numero massimo di stati CI generati classicamente da includere in ciascuna iterazione.                                                                                                                                                                                                                     | Sì      | N/A                                                                      | `10`                                                                                      |
| molecule_options                | dict                                                           | Opzioni relative alla molecola usata come input per HI-VQE. Consulta la sezione [Opzioni molecola](#molecule-options) per ulteriori dettagli.                                                                                                                                                                                                                                                | No       | Consulta la sezione [Opzioni molecola](#molecule-options) per i dettagli.                                 | `{"basis": "sto3g", "unit": "angstrom" }`                               |
| hivqe_options                | dict                                                           | Opzioni che controllano il comportamento dell'algoritmo HI-VQE. Consulta la sezione [Opzioni HI-VQE](#hi-vqe-options) per ulteriori dettagli.                                                                                                                                                                                                                                                | No       | Consulta la sezione [Opzioni HI-VQE](#hi-vqe-options) per i dettagli.                                 | `{"shots": 10_000, "max_iter": 10 }`                               |

### Opzioni molecola
La tabella seguente descrive tutte le chiavi e i valori che possono essere impostati nel dizionario `molecule_options`, con i relativi tipi di dati e valori predefiniti. Tutte le chiavi sono opzionali.

| Chiave                            | Tipo valore                         | Valore predefinito               | Intervallo valido                                                                                                                                                    | Spiegazione                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"charge"`                        | `int`                               | `0`                              | Vario                                                                                                                                                        | Un intero che specifica la carica netta totale del sistema molecolare. Il valore predefinito è 0; tuttavia, può essere qualsiasi intero.                                                                                                                                                                                                                                                                                                                                                                                                              |
| `"basis"`                         | `str`                               | `'sto-3g'`                       | Vario                                                                                                                                                        | Una stringa che specifica il tipo di base; vengono passate a pyscf. (es.: `"sto-3g"`, `"3-21g"`, `"6-31g"`, `"cc-pvdz"`)                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"active_orbitals"`               | `List[int]`                         | Tutti gli indici degli orbitali.             | Gli indici degli orbitali spaziali validi per il problema.                                                                                                             | Una lista di indici degli orbitali attivi nell'intervallo [0, n) dove n è il numero di qubit usati nel problema. Se specificato, l'argomento frozen_orbitals deve essere anch'esso specificato.                                                                                                                                                                                                                                                                                                                                            |
| `"frozen_orbitals"`               | `List[int]`                         | Nessun indice.                      | Gli indici degli orbitali spaziali validi per il problema, esclusi gli orbitali attivi.                                                                                  | Una lista di indici degli orbitali congelati nello stesso intervallo di active_orbitals. Se specificato, active_orbitals deve essere anch'esso specificato. Nota che solo gli orbitali occupati dovrebbero essere congelati, poiché il numero di elettroni attivi viene ridotto di 2 per ogni orbitale occupato congelato.                                                                                                                                                                                                                                        |
| `"orbital_coeffs"`                | `List[List[float]]`                 | Orbitali molecolari Hartree-Fock. | Vario.                                                                                                                                                       | I coefficienti degli orbitali spaziali usati nel calcolo degli integrali di repulsione elettronica del sistema. Alcuni esempi validi sono gli orbitali molecolari Hartree-Fock, gli orbitali naturali e gli orbitali AVAS.                                                                                                                                                                                                                                                                                                                   |
| `"symmetry"`                      | `Union[str, bool]`                  | `False`                          | `True` o `False`                                                                                                                                              | Usato per invocare la simmetria del gruppo puntuale per i calcoli molecolari iniziali, al fine di costruire la base di orbitali adattati per simmetria. Questi orbitali adattati per simmetria sono usati come funzioni di base per i successivi calcoli SCF. Il valore predefinito è False; se impostato a True, verrà invocato e i gruppi puntuali arbitrari verranno rilevati e usati automaticamente. Se viene assegnata una simmetria specifica, ad esempio symmetry = "Dooh", verrà sollevato un errore se la geometria molecolare non soddisfa tale simmetria richiesta. |
| `"symmetry_subgroup"`             | `Optional[str]`                     | `None`                           | Consulta la [documentazione di pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Può essere usato per generare un sottogruppo della simmetria rilevata. Non ha effetto quando la simmetria è specificata tramite l'argomento chiave symmetry.                                                                                                                                                                                                                                                                                                                                                                                                                                 |
| `"unit"`                          | `str`                               | `"angstrom"`                     | Consulta la [documentazione di pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Specifica l'unità di misura da usare per le coordinate atomiche e le distanze. Il valore predefinito è angstrom.                                                                                                                                                                                                                                                                                                                                                                                                                       |
| `"nucmod"`                        | `Optional[Union[dict, str]]`        | `None`                           | Consulta la [documentazione di pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Specifica il modello nucleare da usare. Per impostazione predefinita usa il modello nucleare puntiforme; altri valori abilitano il modello nucleare gaussiano. Se viene fornita una funzione, sarà usata con il modello nucleare gaussiano per generare il valore della distribuzione della carica nucleare 'zeta'.                                                                                                                                                                                                                                   |
| `"pseudo"`                        | `Optional[Union[dict, str]]`        | `None`                           | Consulta la [documentazione di pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Specifica lo pseudopotenziale per gli atomi della molecola. Il valore predefinito è None, il che indica che non vengono applicati pseudopotenziali e tutti gli elettroni sono inclusi esplicitamente nei calcoli.                                                                                                                                                                                                                                                                                                                                                |
| `"cart"`                          | `bool`                              | `False`                          | Consulta la [documentazione di pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Specifica se usare GTO cartesiani come funzioni di base del momento angolare nel calcolo. Il valore predefinito False usa GTO sferici.                                                                                                                                                                                                                                                                                                                                               |
| `"magmom"`                        | `Optional[List[Union[int, float]]]` | `None`                           | Consulta la [documentazione di pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Imposta il momento magnetico di spin colineare di ciascun atomo. Per impostazione predefinita è None e ogni atomo viene inizializzato con spin zero.                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"avas_aolabels"`                 | `Optional[List[str]]`               | `None`                           | es. ["H 1s", "O 2p"] per H2O                                                                                                                                                             | Definisce gli orbitali atomici da includere nello schema AVAS. Consulta la [documentazione AVAS](https://arxiv.org/abs/1701.07862).                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"avas_threshold"`                | `float`                             | `0.2`                            | Tra 0.0 e 2.0                                                                                                                                                      | Specifica il valore soglia usato per determinare quali orbitali atomici (AO) vengono mantenuti nello spazio attivo.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"noons_level"`                   | `Optional[str]`                     | `None`                           | `"mp2"` o `"ccsd"`                                                                                                                                            | Definisce l'approccio teorico per la preparazione degli orbitali naturali e la selezione degli orbitali attivi basata sullo schema dei Natural Orbital Occupation Numbers (NOONs). Consulta la [documentazione NOONs](https://doi.org/10.1063/5.0042147). Sia gli indici degli orbitali attivi che quelli congelati devono essere forniti per controllare il numero di orbitali (e il numero di qubit).                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             |

### Opzioni HI-VQE
La tabella seguente descrive tutte le chiavi e i valori che possono essere impostati nel dizionario `hivqe_options`, con i relativi tipi di dati e valori predefiniti. Tutte le chiavi sono opzionali.

| Chiave                            | Tipo valore                         | Valore predefinito               | Intervallo valido                                                                                                                                                    | Spiegazione                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"shots"`                         | `int`                               | `1_000`                          | Tra 1 e 10 000.                                                                                                                                          | Numero di shot da usare sul dispositivo quantistico per iterazione.                                                                                                                                                                                                                                                                                                                                                                                                                                                             |
| `"max_iter"`                      | `int`                               | `25`                             | Tra 1 e 50.                                                                                                                                              | Il numero massimo di iterazioni da eseguire per ottimizzare l'ansatz. L'algoritmo potrebbe usare meno iterazioni se la convergenza viene raggiunta prima.                                                                                                                                                                                                                                                                                                                                                                                                                 |
| `"initial_basis_states"`          | `List[str]`                         | Lo stato Hartree-Fock.          | Stringhe binarie con il numero di bit corrispondente al numero di qubit richiesti per il problema.                                                                 | Può essere usato per riavviare l'algoritmo con stati classici da un risultato precedente.                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"ansatz"`                        | `str`                               | `"epa"`                          | `"epa"`, `"hea"` o `"lucj"`                                                                                                                                  | Specifica l'ansatz quantistico da ottimizzare per generare nuovi stati. `"epa"` seleziona l'ansatz excitation-preserving. `"hea"` seleziona l'ansatz hardware-efficient. `"lucj"` seleziona l'ansatz local unitary cluster Jastrow.                                                                                                                                                                                                                                                                                                       |
| `"convergence_count"`             | `int`                               | `3`                              | Almeno 2.                                                                                                                                                    | Il numero di iterazioni senza un cambiamento significativo dell'energia calcolata che devono trascorrere prima che l'algoritmo sia considerato convergente.                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"convergence_abstol"`            | `float`                             | `1e-4`                           | Maggiore di 0 e al massimo 1.                                                                                                                                     | L'entità del cambiamento nell'energia calcolata considerata significativa ai fini dei controlli di convergenza.                                                                                                                                                                                                                                                                                                                                                                                                                       |
| `"reset_convergence_count"`       | `bool`                              | `True`                           | `True` o `False`                                                                                                                                              | Se `True`, le iterazioni `convergence_count` devono verificarsi senza un cambiamento significativo che le interrompa per qualificarsi come convergenti. Se `False`, l'algoritmo si fermerà dopo `convergence_count` se sono avvenuti cambiamenti insignificanti in qualsiasi iterazione durante il processo di ottimizzazione.                                                                                                                                                                                                                                                 |
| `"configuration_recovery"`        | `bool`                              | `True`                           | `True` o `False`.                                                                                                                                             | Se usare o meno il configuration recovery dal pacchetto `qiskit-addon-sqd`. Se True, gli stati non validi campionati dal dispositivo quantistico vengono corretti classicamente. Se False, vengono scartati.                                                                                                                                                                                                                                                                                                                                      |
| `"ansatz_entanglement"`           | `str`                               | `"circular"`                     | Una qualsiasi tra `"linear"`, `"reverse_linear"`, `"pairwise"`, `"circular"`, `"full"` o `"sca"`. Se si usa l'ansatz `"lucj"`, è disponibile anche `"lucj_default"`. | Specifica lo schema di entanglement da usare nel circuito quantistico, seguendo le convenzioni comuni di Qiskit e [ffsim per l'ansatz LUCJ](https://qiskit-community.github.io/ffsim/how-to-guides/simulate-lucj.html).                                                                                                                       |
| `"ansatz_reps"`                   | `int`                               | `2`                              | Maggiore di 0.                                                                                                                                                | Il numero di ripetizioni di ciascun layer nel circuito quantistico.                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"amplitude_screening_tolerance"` | `Union[float,int]`                  | `0`                              | Almeno 0 e minore di 1.                                                                                                                                   | La tolleranza per decidere quali stati escludere dal sottospazio dopo la diagonalizzazione. Specifica la soglia di inclusione per gli stati del sottospazio in base alle loro ampiezze calcolate.                                                                                                                                                                                                                                                                                                                                  |
| `"overlap_screening_tolerance"`   | `float`                             | `1e-2`                           | Tra `1e-4` e `1e-1`, inclusi.                                                                                                                          | La tolleranza per prevedere quali stati escludere dal sottospazio prima della diagonalizzazione. Controlla la precisione delle ampiezze previste per ciascuno stato: un valore più piccolo produce previsioni più accurate.                                                                                                                                                                                                                                                                                                            |

## Output
La funzione restituisce un dizionario con quattro chiavi e valori. Le chiavi e i valori sono documentati nella tabella seguente:

| Chiave             | Tipo valore    | Spiegazione                                                                                                               |
|:----------------|---------------|---------------------------------------------------------------------------------------------------------------------------|
| `"energy"`      | `float`       | L'energia approssimata dello stato fondamentale della molecola.                                                                      |
| `"states"`      | `List[str]`   | I determinanti selezionati che formano il sottospazio usato per risolvere l'energia. Sono in formato alpha-beta alternato. |
| `"eigenvector"` | `List[float]` | L'autovettore corrispondente allo stato fondamentale del sottospazio composto da `"states"`.                                 |
| `"energy_variance"` | `float` | La varianza dell'energia dello stato fondamentale del sottospazio composto da `"states"`, che indica la qualità della soluzione. Questo valore è non negativo e un valore inferiore significa che lo stato fondamentale del sottospazio approssima più da vicino un autostato dell'Hamiltoniana del sistema. |
| `"energy_history"` | `List[float]` | Le energie calcolate ad ogni iterazione durante il processo di ottimizzazione ibrido, nell'ordine in cui sono state calcolate. Vengono calcolate due energie per iterazione come parte del processo di ottimizzazione SPSA. |

## Esempio
Il primo esempio mostra come calcolare l'energia dello stato fondamentale per una molecola di NH3 usando l'algoritmo HI-VQE.

#### Definire la geometria molecolare e le opzioni
La geometria molecolare di NH3 viene fornita con coordinate cartesiane separate da ";" per ciascun atomo.

In [3]:
# Define the molecule geometry
geometry = """
N         -0.85188       -0.02741        0.03141;
H          0.16545        0.00593       -0.01648;
H         -1.16348       -0.39357       -0.86702;
H         -1.16348        0.94228        0.06281;
"""

È possibile definire e fornire opzioni aggiuntive per il sistema molecolare nel seguente formato dizionario.

In [4]:
# Configure some options for the job.
molecule_options = {"basis": "sto3g"}
hivqe_options = {"shots": 100, "max_iter": 20}

Esegui la funzione con la geometria e le opzioni come input.

In [5]:
# Run HI-VQE
job = function.run(
    geometry=geometry,
    # `backend_name` is the name of a backend with at least 16 qubits, for example, "ibm_marrakesh".
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=10,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

È buona pratica stampare l'ID del job della Function in modo da poterlo fornire nelle richieste di supporto in caso di problemi.

In [6]:
print("Job ID:", job.job_id)

Job ID: 128ee399-7cfc-4be2-91e9-c4abd22b97c7


This example then utilizes 16 qubits with 8 orbitals of sto3g basis for an NH3 molecule.

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [7]:
print(job.status())

QUEUED


Questo esempio utilizza 16 qubit con 8 orbitali di base sto3g per una molecola di NH3.
Controlla lo [stato](/guides/functions#check-job-status) del workload della tua Qiskit Function o recupera i [risultati](/guides/functions#retrieve-results) come segue:

In [8]:
result = job.result()

# Output can be long, so we display a shortened representation
shortened_result = reprlib.repr(result)
print(shortened_result)

{'eigenvector': [0.9824200343205695, 0.009520846167419264, 6.365368845740147e-08, 3.6832123006425615e-07, 0.0012869941182066654, 2.3403891050875465e-05, ...], 'energy': -55.521146537970466, 'energy_history': [-55.52091922342852, -55.52113695367561, -55.521146537970466, -55.52114653864798, -55.521146537970466, -55.521146537970466, ...], 'energy_variance': 9.788555455342562e-12, ...}


To access the ground state energy, use the "energy" key. The "eigenvector" key provides the CI coefficients with corresponding bitstring notation of electron configuration stored with "states" of the results.

In [10]:
fci_energy = -55.521148034704126  # the exact energy using FCI method
hivqe_energy = result["energy"]
print(
    f"|Exact Energy - HI-VQE Energy|: {abs(fci_energy - hivqe_energy) * 1000} mHa"
)
print(f"Sampled Number of States: {len(result['states'])}")

|Exact Energy - HI-VQE Energy|: 0.0014967336596782843 mHa
Sampled Number of States: 1936


Una volta completato il job, i risultati possono essere ottenuti con l'istanza `result()`.

In [11]:
# This cell is hidden from users
backend_name = service.least_busy(operational=True, min_num_qubits=38).name

In [12]:
# Define Li2S geometries
Li2S_geoms = {
    "Li2S_1.51": "S        -1.239044    0.671232   -0.030374;Li       -1.506327    0.432403   -1.498949;Li       -0.899996    0.973348    1.826768;",
    "Li2S_2.40": "S        -1.741432    0.680397    0.346702;Li       -0.529307    0.488006   -1.729343;Li       -1.284307    0.989409    2.177209;",
    "Li2S_3.80": "S        -2.707255    0.674298    0.909161;Li        0.079218    0.552012   -1.671656;Li       -0.927010    0.931502    1.557063;",
}

# Configure some options for the job.
molecule_options = {
    "basis": "sto3g",
}
hivqe_options = {
    "shots": 100,
    "max_iter": 20,
}

results = []
for geom in ["Li2S_1.51", "Li2S_2.40", "Li2S_3.80"]:
    # Run HI-VQE
    job = function.run(
        geometry=Li2S_geoms[geom],
        backend_name=backend_name,  # can use any device with at least 38 qubits
        max_states=2000,
        max_expansion_states=10,
        molecule_options=molecule_options,
        hivqe_options=hivqe_options,
    )
    results.append(job.result())

Per accedere all'energia dello stato fondamentale, usa la chiave `"energy"`. La chiave `"eigenvector"` fornisce i coefficienti CI con la corrispondente notazione a stringa di bit della configurazione elettronica memorizzata in `"states"` dei risultati.

In [13]:
job = function.run(
    geometry="invalid-geometry",  # This will cause an error
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=15,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

job.result()

QiskitServerlessException: {"message": "An unexpected error occurred during job execution. Please make sure that your inputs are valid. If you are still experiencing problems, you can contact the Qunova Computing support service at qiskit.support@qunovacomputing.com and provide the Function job ID of this job for more assistance.", "status": "failure"}

In [14]:
job.status()

'ERROR'

Output:

|Exact Energy - HI-VQE Energy|: 0.077237504 mHa
Sampled Number of States: 1936

## Prestazioni
Questa sezione mostra i calcoli di benchmark dimostrati di HI-VQE con un caso a 24 qubit per Li2S, un caso a 40 qubit per una molecola di N2 e un caso a 44 qubit per un sistema FeP-NO.

#### Curva della superficie di energia potenziale di dissociazione per una molecola di Li2S con 24 qubit
La curva PES è mostrata con il riferimento FCI e la stima iniziale da RHF, insieme all'errore di energia rispetto al riferimento FCI.

![Immagine che mostra che HI-VQE produce soluzioni entro la precisione chimica della curva PES di riferimento classica per il sistema Li2S.](../docs/images/guides/qunova-chemistry/Li2S_PES.avif)

I calcoli sono stati condotti con le seguenti geometrie e opzioni.